# Trijunction curves from the mask, keyed to the mesh's own junctions

`dw3d.extract_junction_curves` reads the trijunction curves off the segmentation mask and matches
them against the trijunctions the reconstructed mesh itself reports.

This notebook reconstructs `data/Images/3.tif`, extracts the curves, and prints two things:

* the **match rate** — how many of the mesh's own trijunctions have exactly one extracted counterpart;
* **L5** — the median distance, in voxels, between each matched curve and the mesh's own line for
  that same material triple.

L5 here is a *disagreement*, not an error: no ground truth is read anywhere in this notebook.
Which of the two is closer to the truth was measured separately, against a registered reference on
a 47-case synthetic benchmark, and the extraction wins by 1.48× at `min_distance = 3` — see
`BENCHMARKS.md`, section "Junction curves".

Runs in well under a minute and needs nothing outside this repository.


In [ ]:
from pathlib import Path
import time

import numpy as np
import skimage.io as io

from dw3d import extract_junction_curves, get_default_mesh_reconstruction_algorithm
from dw3d.junction_curves import mesh_trijunction_components

IMAGE = Path("../data/Images/3.tif")
MIN_DISTANCE = 3

mask = io.imread(IMAGE)
print(f"mask {IMAGE.name}: shape {mask.shape}, {len(np.unique(mask))} materials")

## 1) Reconstruct the mesh, exactly as you normally would

In [ ]:
algorithm = get_default_mesh_reconstruction_algorithm(min_distance=MIN_DISTANCE)

started = time.perf_counter()
points, triangles, labels = algorithm.construct_mesh_from_segmentation_mask(mask)
print(f"{len(points)} points, {len(triangles)} triangles, {time.perf_counter() - started:.1f} s")

## 2) The one line

The reading spacing is not a parameter: the polylines come back resampled at the mesh's **own**
median trijunction-edge length, so the two descriptions of the same line are read at the same
resolution.

In [ ]:
started = time.perf_counter()
curves = extract_junction_curves(mask, points, triangles, labels)
print(f"{time.perf_counter() - started:.1f} s, estimator {curves.estimator!r}, "
      f"read at the mesh's own {curves.spacing_voxels:.2f}-voxel junction spacing")

## 3) What came back

`matched` is the usable set. `invented` is kept separate on purpose: on the benchmark, **none** of
the invented curves lies within a voxel of all three of its own interfaces, where 78 % of matched
curves do. They are two different populations, not two ends of one distribution.

In [ ]:
print(f"match rate: {curves.match_rate:.3f}  "
      f"({curves.counts['matched']} of the mesh's {curves.counts['mesh_triples']} trijunctions)")
print(f"  matched    {len(curves.matched):4d} curves  <- the usable set")
print(f"  fragmented {len(curves.fragmented):4d} curves  <- real trijunctions read as several pieces")
print(f"  invented   {len(curves.invented):4d} curves  <- no trijunction of that triple in the mesh")
print(f"  missed     {curves.counts['missed']:4d} triples")

first = curves.matched[0]
print(f"\nfirst matched curve: triple {first.triple}, {len(first.points)} points, "
      f"arclength {first.arclength:.2f} voxels")

## 4) How far the two descriptions of the same line sit apart

In [ ]:
def point_to_polyline_distances(query, starts, ends):
    """Exact point-to-segment distance from every point to the nearest of a set of segments."""
    direction = ends - starts
    denominator = (direction * direction).sum(axis=1)
    denominator[denominator == 0] = 1.0
    delta = query[:, None, :] - starts[None, :, :]
    t = np.clip((delta * direction[None, :, :]).sum(axis=2) / denominator[None, :], 0.0, 1.0)
    closest = starts[None, :, :] + t[:, :, None] * direction[None, :, :]
    return np.linalg.norm(query[:, None, :] - closest, axis=2).min(axis=1)


mesh_lines = mesh_trijunction_components(points, triangles, labels)
distances = []
for curve in curves.matched:
    block = mesh_lines[curve.triple][0]
    index_of = {int(v): i for i, v in enumerate(block["vertices"])}
    starts = block["points"][[index_of[v1] for v1, _ in block["edges"]]]
    ends = block["points"][[index_of[v2] for _, v2 in block["edges"]]]
    distances.append(float(np.median(point_to_polyline_distances(curve.points, starts, ends))))

distances = np.asarray(distances)
print("L5, extracted curve against the mesh's own line for the same triple:")
print(f"  median {np.median(distances):.3f} voxels over {len(distances)} matched curves")
print(f"  p90    {np.percentile(distances, 90):.3f} voxels")
print(f"  max    {distances.max():.3f} voxels")
print(f"\nthat is {np.median(distances) / curves.spacing_voxels:.2f} of the mesh's own "
      "junction-edge length")

## What the number means, and what it does not

The two descriptions of the same junction lines differ well below the mesh's own resolution — but
they do differ, and measured against a registered reference the extraction is the closer of the
two (median position error 0.5846 voxels against the mesh's 0.8642 at `min_distance = 3`).

It is still about **4.8×** above the mask's own identifiability floor, which was measured at
0.107 voxels. So this is **better geometry, not correct geometry**.

Two other things worth knowing before you build on it:

* the curves are *less complete* than the mesh's lines — 90 % of the mesh's trijunctions get
  exactly one curve and the rest arrive fragmented, so code consuming them must handle "several
  components for one triple";
* the default estimator runs no linear program. `estimator="anchor"` solves one per vertex, costs
  about six times as much, and buys ~11 % of position error; its argument is a smaller systematic
  bias against narrow junction wedges, not that 11 %.